[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C32_Skills_Tools_Course/05_skills_system/05_skills_system.ipynb)

# 05 · 完整 Skills 系统（capstone）

目标：用**纯标准库**把前四模块的零件——skill 加载、slash 命令、MCP 工具、插件——组装成一个**不依赖框架**的完整 skills 系统，接到 **agent 循环**（MockLLM 驱动），**端到端完成一个真实任务**，并做最基本的**评测**（召回/精度/完成率/token），全程 `assert` 验证、**无需 API key**。

路线：MockLLM + make_llm（无 key 回退）→ 三个注册表（skill/command/tool）→ 选择(selection) → 组合 + 注入 → 工具分发 → **agent 循环端到端完成任务** → 评测 → ✏️ 练习 → 📖 答案 → 🧪 真实 skills 目录胶囊。

> 心智模型：**完整系统 = 三注册表统一在『发现→选择→注入/调用』流水线上 + 接到 agent 循环**。前四模块教你造零件，本模块教你装成机器并发动它。难点在零件之间的接合：相关性怎么判、上下文怎么拼、工具怎么调、循环何时停、做得好不好怎么量。

> 本 notebook **自洽**：它紧凑地重新定义所需的加载器/注册表，不依赖其它 notebook。

## 1 · 本课主角与统一 LLM 工厂（无 key 自动回退）

先把全课共用的 **MockLLM**（确定性假模型）与 **make_llm**（有 `ANTHROPIC_API_KEY` 走真实 Claude、无则回退 MockLLM）搬过来。
capstone 里这个范式尤其重要：整条 agent 循环写好后，把模型从 MockLLM 换成真实 Claude **只改这一处**。

In [ ]:
import os, json, re, shlex

class MockLLM:
    '''确定性假模型：按规则把 prompt/messages 映射到响应。第一个命中的规则生效。
       响应形状贴近真实 tool-use: {'type':'tool_use','name','input'} / {'type':'final','text'}.'''
    def __init__(self, rules, default=None):
        self.rules = rules
        self.default = default or {'type': 'final', 'text': '(no rule matched)'}
        self.calls = 0
    def __call__(self, prompt):
        self.calls += 1
        text = prompt if isinstance(prompt, str) else json.dumps(prompt, ensure_ascii=False)
        for kw, resp in self.rules:
            if kw in text:
                return json.loads(json.dumps(resp))   # 深拷贝
        return json.loads(json.dumps(self.default))

def make_llm(rules=None, default=None, model='claude-sonnet-4-6'):
    '''有 ANTHROPIC_API_KEY 且装了 anthropic -> 真实 Claude；否则 -> MockLLM。绝不阻断。'''
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic
            client = anthropic.Anthropic()
            def real_llm(prompt):
                msgs = prompt if isinstance(prompt, list) else [{'role': 'user', 'content': prompt}]
                resp = client.messages.create(model=model, max_tokens=1024, messages=msgs)
                for b in resp.content:
                    if b.type == 'tool_use':
                        return {'type': 'tool_use', 'name': b.name, 'input': b.input}
                return {'type': 'final', 'text': ''.join(b.text for b in resp.content if b.type == 'text')}
            print(f'[make_llm] 真实 Claude: {model}')
            return real_llm
        except Exception as e:
            print(f'[make_llm] 真实 Claude 不可用({type(e).__name__}), 回退 MockLLM')
    print('[make_llm] 无 API key, 使用 MockLLM')
    return MockLLM(rules or [], default=default)

llm = make_llm(rules=[('提交', {'type': 'tool_use', 'name': 'git_commit', 'input': {'message': 'feat: x'}})])
out = llm('帮我把改动提交了')
print('输出:', out)
assert out['type'] == 'tool_use' and out['name'] == 'git_commit'
assert isinstance(make_llm(), MockLLM)   # 本环境无 key -> MockLLM
print('✅ MockLLM + make_llm 就位：无 key 回退, 有 key 走真实 Claude')

## 2 · skill 注册表：解析 frontmatter + 渐进披露

承接模块 01：一个 skill 是 `---` frontmatter(name/description) + markdown 正文。注册表常驻**轻量目录**(name+描述)，正文**按需加载**。

这里用纯标准库手写一个极简 frontmatter 解析器（不依赖 PyYAML），并把若干 SKILL.md 字符串注册进来。

In [ ]:
def parse_skill(text):
    '''解析 SKILL.md 字符串 -> (meta_dict, body)。frontmatter 为 --- 包裹的 key: value。'''
    meta, body = {}, text
    if text.startswith('---'):
        end = text.find('\n---', 3)
        if end != -1:
            fm = text[3:end].strip()
            body = text[end + 4:].lstrip('\n')
            for line in fm.splitlines():
                if ':' in line:
                    k, v = line.split(':', 1)
                    meta[k.strip()] = v.strip()
    return meta, body

class SkillRegistry:
    def __init__(self):
        self.skills = {}                  # name -> {'desc':..., 'body':...}
    def register(self, text):
        meta, body = parse_skill(text)
        name = meta.get('name')
        assert name, 'skill 必须有 name'
        self.skills[name] = {'desc': meta.get('description', ''), 'body': body}
        return name
    def catalog(self):
        '''常驻轻量目录: name -> 描述(不含正文)。'''
        return {n: s['desc'] for n, s in self.skills.items()}
    def body(self, name):
        '''按需加载某 skill 的正文。'''
        return self.skills[name]['body']

# 注意: description 用空格分隔成『关键词』, 这样朴素的相关性匹配才能按词命中;
# 且关键词应有区分度(避免『写』这种各任务都有的泛词造成误命中, 拉低精度)。
GIT_SKILL = '''---\nname: git\ndescription: git 提交 分支 合并\n---\n提交信息用祈使句, 50 字内。'''
SQL_SKILL = '''---\nname: sql\ndescription: SQL 查询 优化 索引\n---\n先看表结构, 避免 SELECT *。'''
DOCS_SKILL = '''---\nname: docs\ndescription: 文档 润色 大纲\n---\n先列大纲, 每段一个要点。'''

sreg = SkillRegistry()
for t in (GIT_SKILL, SQL_SKILL, DOCS_SKILL):
    sreg.register(t)
print('skill 目录(常驻):', sreg.catalog())
assert set(sreg.catalog()) == {'git', 'sql', 'docs'}
assert '祈使句' in sreg.body('git')                 # 正文按需可取
assert '祈使句' not in json.dumps(sreg.catalog(), ensure_ascii=False)  # 目录不含正文(渐进披露)
print('✅ skill 注册表：frontmatter 解析正确、目录轻量、正文按需')

## 3 · tool 注册表 + 分发

承接模块 03：工具是「name → (可执行函数, schema)」。注册表常驻**工具简介**(渐进披露)，分发器把模型给的 `tool_use` 路由到函数，**任何错误都包成结果而非崩溃**。

In [ ]:
class ToolRegistry:
    def __init__(self):
        self.tools = {}                  # name -> {'fn', 'schema', 'desc'}
    def register(self, name, fn, desc, schema):
        self.tools[name] = {'fn': fn, 'desc': desc, 'schema': schema}
    def specs_text(self):
        '''常驻轻量: 只列 name + 简介(完整 schema 可按需展开)。'''
        return '; '.join(f"{n}({t['desc']})" for n, t in self.tools.items())
    def schema(self, name):
        return self.tools[name]['schema']        # 按需取完整 schema
    def dispatch(self, call):
        '''call = {'name', 'input'} -> 结果字符串(错误也包成结果, 不抛)。'''
        name = call.get('name')
        if name not in self.tools:
            return f'[error] 未知工具: {name}'
        try:
            return str(self.tools[name]['fn'](**call.get('input', {})))
        except Exception as e:
            return f'[error] {type(e).__name__}: {e}'

# 一个玩具仓库状态 + 两个 git 工具
REPO = {'dirty': True, 'committed': None}
def git_status():
    return '有未提交改动' if REPO['dirty'] else '工作区干净'
def git_commit(message):
    if not REPO['dirty']:
        raise ValueError('没有可提交的改动')
    REPO['dirty'] = False; REPO['committed'] = message
    return f'已提交: {message}'

treg = ToolRegistry()
treg.register('git_status', git_status, '查看工作区改动', {'type': 'object', 'properties': {}})
treg.register('git_commit', git_commit, '提交改动并写提交信息',
              {'type': 'object', 'properties': {'message': {'type': 'string'}}, 'required': ['message']})
print('工具简介(常驻):', treg.specs_text())
print(treg.dispatch({'name': 'git_status', 'input': {}}))
print(treg.dispatch({'name': 'rm_rf', 'input': {}}))       # 未知工具 -> 安全包错
assert treg.dispatch({'name': 'git_status', 'input': {}}) == '有未提交改动'
assert treg.dispatch({'name': 'rm_rf', 'input': {}}).startswith('[error]')
print('✅ tool 注册表 + 分发：简介常驻、schema 按需、错误安全包成结果')

## 4 · command 路由（轻量，承接模块 02）

完整系统的第三个注册表是 slash 命令路由。这里给一个**轻量版**：解析 `/cmd args`、把 `$ARGUMENTS` 绑进模板、路由到模板或报『未知命令』。（动态注入 `!shell`/`@file` 在模块 02 已深入，这里聚焦『命令也是一类可发现、可路由的能力』。）

In [ ]:
class CommandRouter:
    def __init__(self):
        self.cmds = {}                   # name -> 模板(含 $ARGUMENTS / $1 占位)
    def register(self, name, template):
        self.cmds[name] = template
    def parse(self, line):
        '''/cmd a "b c" -> (name, [args])。用 shlex 处理带引号参数。'''
        assert line.startswith('/'), 'slash 命令以 / 开头'
        parts = shlex.split(line[1:])
        return parts[0], parts[1:]
    def expand(self, line):
        '''解析并展开成最终 prompt。未知命令安全报错。'''
        name, args = self.parse(line)
        if name not in self.cmds:
            return f'[error] 未知命令: {name}'
        tpl = self.cmds[name]
        tpl = tpl.replace('$ARGUMENTS', ' '.join(args))
        for i, a in enumerate(args, 1):
            tpl = tpl.replace(f'${i}', a)
        return tpl

creg = CommandRouter()
creg.register('commit', '请用 git skill 的规范, 把改动提交, 提交信息主题: $ARGUMENTS')
name, args = creg.parse('/commit "修复登录 bug"')
print('解析:', name, args)
print('展开:', creg.expand('/commit "修复登录 bug"'))
assert name == 'commit' and args == ['修复登录 bug']
assert '修复登录 bug' in creg.expand('/commit "修复登录 bug"')
assert creg.expand('/unknown').startswith('[error]')
print('✅ command 路由：解析带引号参数、$ARGUMENTS 绑定、未知命令安全报错')

## 5 · 组装成 SkillSystem：选择 + 注入

现在把三个注册表统一进一个 **SkillSystem**，实现流水线的核心两步：**选择**(按相关性挑出相关 skill)与**注入**(只把命中 skill 的正文 + 工具简介拼进上下文)。

选择是把『潜在能力』收敛成『本轮实际能力』；注入是渐进披露的落地——未命中的正文绝不进上下文。

In [ ]:
class SkillSystem:
    '''统一三个注册表 + 流水线(发现已在注册时完成; 这里做选择/注入/调用)。'''
    def __init__(self, skills, tools, commands):
        self.skills = skills            # SkillRegistry
        self.tools = tools              # ToolRegistry
        self.commands = commands        # CommandRouter
    def select_skills(self, task):
        '''选择: 任务命中某 skill 的 description 关键词 -> 选中。返回 name 列表(有序)。'''
        hits = []
        for name, desc in self.skills.catalog().items():
            kws = desc.replace(',', ' ').replace('、', ' ').split()
            if any(kw in task for kw in kws):
                hits.append(name)
        return hits
    def inject(self, skill_names):
        '''注入: 只取命中 skill 的正文(渐进披露)。'''
        return '\n'.join(f'[skill:{n}] {self.skills.body(n)}' for n in skill_names)
    def build_context(self, task):
        '''选择 + 注入 + 提供工具简介, 拼成发给模型的上下文。'''
        names = self.select_skills(task)
        parts = []
        if names:
            parts.append(self.inject(names))
        parts.append('可用工具: ' + self.tools.specs_text())
        parts.append('任务: ' + task)
        return '\n'.join(parts), names

system = SkillSystem(sreg, treg, creg)
ctx, names = system.build_context('帮我把改动提交了')
print('选中 skill:', names)
print('--- 上下文 ---')
print(ctx)
assert names == ['git'], '只有 git skill 应被任务『提交』命中'
assert '祈使句' in ctx                                  # git 正文被注入
assert 'SELECT' not in ctx and '大纲' not in ctx        # sql/docs 正文未注入(渐进披露!)
assert 'git_commit' in ctx                              # 工具简介在上下文里(模型才知道有手)
print('✅ SkillSystem 选择+注入：选中相关 skill、注入其正文、提供工具、未命中不进上下文')

## 6 · agent 循环：端到端用 skill + 工具完成任务

最后一步：把 SkillSystem 接到 **agent 循环**，让 MockLLM 驱动一次真实任务的**组合执行**——
任务『把改动提交了』→ 选中 git skill + 注入 → 模型先调 `git_status` 看改动 → 再调 `git_commit` 写规范提交信息 → 模型确认完成。

循环必须**确定性且终止**（max_steps 守卫）。这正是模块 00 那条流水线『注入/调用』段被反复执行。

In [ ]:
def agent_run(task, system, llm, max_steps=6):
    '''完整 skills 系统的 agent 循环。返回 (最终答复, 选中skill, 调用过的工具名列表)。'''
    ctx, selected = system.build_context(task)
    messages = [{'role': 'user', 'content': ctx}]
    tools_called = []
    for step in range(max_steps):
        resp = llm(messages)
        if resp['type'] == 'final':
            return resp['text'], selected, tools_called
        # tool_use: 分发执行(组合的一步), 结果回喂
        tools_called.append(resp['name'])
        result = system.tools.dispatch(resp)
        messages.append({'role': 'assistant', 'content': resp})
        messages.append({'role': 'user', 'content': f'工具结果: {result}'})
    return None, selected, tools_called          # 超步数也要停

# MockLLM 规则: 驱动一次多步组合(看状态 -> 提交 -> 完成)
# 注意规则顺序: 先匹配更具体的『工具结果: 已提交』, 再匹配『有未提交改动』, 最后任务起步。
loop_llm = MockLLM(rules=[
    ('已提交',       {'type': 'final', 'text': '改动已按规范提交完成。'}),         # 看到提交成功 -> 收尾
    ('有未提交改动', {'type': 'tool_use', 'name': 'git_commit',
                     'input': {'message': '修复登录 bug'}}),                       # 看到有改动 -> 提交
    ('任务',         {'type': 'tool_use', 'name': 'git_status', 'input': {}}),     # 任务起步 -> 先看状态
])

REPO['dirty'] = True; REPO['committed'] = None        # 复位仓库
answer, selected, called = agent_run('帮我把改动提交了', system, loop_llm)
print('最终答复:', answer)
print('选中 skill:', selected)
print('调用工具序列:', called)
print('仓库状态: committed =', REPO['committed'])
assert selected == ['git'], '应选中 git skill'
assert called == ['git_status', 'git_commit'], '应先看状态再提交(组合多步)'
assert REPO['committed'] == '修复登录 bug', '提交应真的发生且信息正确'
assert answer == '改动已按规范提交完成。'
assert loop_llm.calls <= 6, '必须在 max_steps 内终止'
print('✅ 端到端：选 skill→注入→看状态→提交→收尾, 任务真正完成且循环终止')

## 7 · 评测：召回 / 精度 / 完成率 / token

系统造好了, 好不好用**数字**说话。在一个小标注任务集上, 算出**选择的召回与精度**、**任务完成率**、**token(上下文字符)用量**。

$\text{recall}=|S\cap R|/|R|$、$\text{precision}=|S\cap R|/|S|$, 其中 $R$ 是任务真正相关的 skill, $S$ 是系统选中的。

In [ ]:
def evaluate(system, tasks):
    '''tasks = [(task, set_of_relevant_skill_names)]; 返回平均召回/精度/完成率/平均上下文长度。'''
    recalls, precisions, ctx_lens = [], [], []
    for task, R in tasks:
        ctx, S = system.build_context(task)
        S = set(S)
        inter = S & R
        recalls.append(len(inter) / len(R) if R else 1.0)
        precisions.append(len(inter) / len(S) if S else 1.0)
        ctx_lens.append(len(ctx))
    n = len(tasks)
    return {'recall': sum(recalls) / n, 'precision': sum(precisions) / n,
            'avg_ctx_chars': sum(ctx_lens) / n}

EVAL_TASKS = [
    ('帮我把改动提交了, 写个提交信息', {'git'}),
    ('优化这条 SQL 查询', {'sql'}),
    ('润色一下这段技术文档', {'docs'}),
]
metrics = evaluate(system, EVAL_TASKS)
print('评测结果:', {k: round(v, 3) for k, v in metrics.items()})
assert metrics['recall'] == 1.0, '三个任务的相关 skill 都应被选中(召回=1)'
assert metrics['precision'] == 1.0, '不应选中无关 skill(精度=1)'
# 渐进披露的收益: 对比『按需注入』与『把所有 skill 正文全注入』的上下文长度。
def full_inject_ctx(system, task):
    '''反例基线: 不做选择, 把每个 skill 的正文全部注入。'''
    allnames = list(system.skills.catalog())
    body = system.inject(allnames)
    return body + '\n可用工具: ' + system.tools.specs_text() + '\n任务: ' + task
full_len = sum(len(full_inject_ctx(system, t)) for t, _ in EVAL_TASKS) / len(EVAL_TASKS)
print(f'按需注入平均: {metrics["avg_ctx_chars"]:.1f} 字符 | 全量注入平均: {full_len:.1f} 字符')
assert metrics['avg_ctx_chars'] < full_len, '渐进披露应比全量注入更省上下文'
print('✅ 评测：召回/精度=1, 按需注入比全量注入省上下文 —— 四个维度表现达标')

---
## ✏️ 练习 1：组装一个新能力（注册 + 选择）

给系统加一个新 skill。实现 `add_skill_and_check(system, skill_text, task)`：把 `skill_text` 注册进 `system.skills`，然后对 `task` 跑选择，返回 `(新skill名, 该skill是否被task选中)`。

用来验证『新能力一旦注册，就能在相关任务里被自动选中』——这正是可扩展性的核心。

In [ ]:
def add_skill_and_check(system, skill_text, task):
    # TODO: 1) name = system.skills.register(skill_text)
    #       2) selected = system.select_skills(task)
    #       3) 返回 (name, name in selected)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
TEST_SKILL = '''---\nname: test\ndescription: 单元测试 测试 编写\n---\n用 arrange-act-assert 三段式, 一个测试只验一件事。'''
nm, hit = add_skill_and_check(system, TEST_SKILL, '帮我写个单元测试')
assert nm == 'test'
assert hit is True, '新注册的 test skill 应被『单元测试』任务选中'
# 且不影响无关任务
assert 'test' not in system.select_skills('优化 SQL')
print('✅ 练习 1 通过：新能力注册后即可被相关任务自动选中(可扩展性)')

## ✏️ 练习 2：多 skill 命中时的 token 预算控制

当一个任务同时命中多个 skill, 注入的正文加起来可能超预算。实现 `inject_within_budget(system, skill_names, budget)`：

按给定顺序注入命中 skill 的正文(`[skill:name] 正文` 每条占一行), **但累计字符数不得超过 `budget`**——一旦加上下一条会超预算, 就停止(不再加更多)。返回注入字符串。

In [ ]:
def inject_within_budget(system, skill_names, budget):
    # TODO: 逐个 skill 取 system.skills.body(name), 拼成 '[skill:name] 正文';
    #       维护累计长度, 一旦『加上这一条后』会超 budget 就停止, 返回已拼接部分。
    #       (提示: 用 '\n' 连接已收集的行; 判断 len(候选拼接) <= budget)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
names3 = ['git', 'sql', 'docs']            # 三个都注入会较长
full = system.inject(names3)
tight = inject_within_budget(system, names3, budget=40)
print('完整注入长度:', len(full), '| 预算内长度:', len(tight))
assert len(tight) <= 40, '不得超预算'
assert tight.startswith('[skill:git]'), '应按顺序先注入 git'
assert len(tight) < len(full), '预算应当截断掉部分 skill'
# 预算极小: 一条都放不下时返回空
assert inject_within_budget(system, names3, budget=1) == ''
print('✅ 练习 2 通过：多 skill 命中时按预算截断, 渐进披露的最后一道闸')

## ✏️ 练习 3：端到端任务完成判定

评测里最重要的是『任务到底完成没有』。实现 `task_completed(task, system, llm, check, max_steps=6)`：

跑 `agent_run`, 然后用回调 `check()`(返回 bool, 检查世界状态, 如 `REPO['committed']` 是否被正确设置)判断任务是否真完成。返回 `(是否完成, 调用过的工具数)`。

In [ ]:
def task_completed(task, system, llm, check, max_steps=6):
    # TODO: 1) answer, selected, called = agent_run(task, system, llm, max_steps)
    #       2) done = bool(check())   # check() 检查世界状态
    #       3) 返回 (done, len(called))
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
REPO['dirty'] = True; REPO['committed'] = None      # 复位
llm3 = MockLLM(rules=[
    ('已提交',       {'type': 'final', 'text': '完成。'}),
    ('有未提交改动', {'type': 'tool_use', 'name': 'git_commit', 'input': {'message': 'fix: x'}}),
    ('任务',         {'type': 'tool_use', 'name': 'git_status', 'input': {}}),
])
done, n_calls = task_completed('帮我把改动提交了', system, llm3,
                               check=lambda: REPO['committed'] == 'fix: x')
assert done is True, '提交应真的发生, 任务完成'
assert n_calls == 2, '应调用了 git_status + git_commit 两个工具'
print('✅ 练习 3 通过：用世界状态判定任务真完成(端到端完成率的基础)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def add_skill_and_check(system, skill_text, task):
    name = system.skills.register(skill_text)
    selected = system.select_skills(task)
    return name, name in selected

In [ ]:
# 练习 2 参考答案
def inject_within_budget(system, skill_names, budget):
    lines = []
    for name in skill_names:
        line = f'[skill:{name}] {system.skills.body(name)}'
        candidate = '\n'.join(lines + [line])
        if len(candidate) <= budget:
            lines.append(line)
        else:
            break
    return '\n'.join(lines)

In [ ]:
# 练习 3 参考答案
def task_completed(task, system, llm, check, max_steps=6):
    answer, selected, called = agent_run(task, system, llm, max_steps)
    return bool(check()), len(called)

---
## 🧪 真实数据胶囊：一个真实形状的 skills 目录

下面是一个**贴近真实**的 mini skills 目录(形如 Claude Code 的 `.claude/skills/<name>/SKILL.md`)——三个 skill 文件 + 两个工具。我们让完整系统在这个目录上跑一个任务, 体会真实 skills 系统的样子。

> 形状对照：真实里每个 skill 是一个文件夹含 `SKILL.md`(frontmatter `name`/`description` + 正文)；本课用字符串模拟文件内容, 解析与选择逻辑与真实一致。

In [ ]:
# 真实形状的 SKILL.md(贴近 Anthropic Agent Skills / Claude Code 约定)
REAL_SKILLS_DIR = {
    'code-review/SKILL.md':
        '---\nname: code-review\ndescription: 代码 审查 bug 评审\n---\n'
        '按『正确性 > 可读性 > 性能』排序问题; 每条给文件:行号和具体建议。',
    'changelog/SKILL.md':
        '---\nname: changelog\ndescription: 变更日志 changelog 变更\n---\n'
        '按 Added/Changed/Fixed 分组, 面向用户描述, 不堆术语。',
    'release/SKILL.md':
        '---\nname: release\ndescription: 发布 tag 语义化版本\n---\n'
        '先跑测试, 再打语义化版本 tag, 发布说明引用 changelog。',
}

# 用完整系统加载这个真实目录
real_sys = SkillSystem(SkillRegistry(), ToolRegistry(), CommandRouter())
for path, content in REAL_SKILLS_DIR.items():
    real_sys.skills.register(content)          # 解析 frontmatter 并注册
print('从真实目录发现的 skill:', list(real_sys.skills.catalog()))
# 一个任务: 整理变更日志(刻意不含『发布』, 以免也命中 release —— 体会关键词选择的边界)
ctx, sel = real_sys.build_context('帮我整理这次的变更日志')
print('任务选中:', sel)
assert set(real_sys.skills.catalog()) == {'code-review', 'changelog', 'release'}
assert sel == ['changelog'], '『变更日志』任务应选中 changelog skill'
assert 'Added/Changed/Fixed' in ctx              # changelog 正文被注入
assert '语义化版本' not in ctx                    # release 正文未注入(渐进披露)
print('✅ 真实 skills 目录：发现三个 skill、任务选中正确、渐进披露生效')

**🧪 胶囊练习**：实现 `discover_count(skills_dir)`：给定一个 `{路径: SKILL.md内容}` 的目录字典, 返回成功解析出 `name` 的 skill 数量。(真实里『扫描目录、统计有效 skill』就是这么做的。)

In [ ]:
def discover_count(skills_dir):
    # TODO: 对每个 content 调 parse_skill, 统计 meta 里有非空 'name' 的数量
    raise NotImplementedError

In [ ]:
# 自测
n = discover_count(REAL_SKILLS_DIR)
assert n == 3
# 残缺(无 name)的不计入
broken = {'x/SKILL.md': '---\ndescription: 没有 name\n---\n正文'}
assert discover_count(broken) == 0
print('发现有效 skill 数:', n)
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def discover_count(skills_dir):
    cnt = 0
    for content in skills_dir.values():
        meta, _ = parse_skill(content)
        if meta.get('name'):
            cnt += 1
    return cnt

---
## 🔧 旁注：把整条 agent 循环换上真实 Claude

本课用 MockLLM 跑通的完整系统, 换成真实 Claude **只改模型这一处**——`agent_run` 的 scaffold、三个注册表、选择/注入/分发逻辑**全部原样适用**：

```python
# 真实版(本环境不跑, 需 ANTHROPIC_API_KEY; 无 key 自动回退 MockLLM)
import anthropic
client = anthropic.Anthropic()                     # 读 ANTHROPIC_API_KEY

def agent_run_real(task, system, model='claude-sonnet-4-6', max_steps=6):
    ctx, selected = system.build_context(task)
    # 把 tool_registry 的 schema 直接喂给真实 API(形状一致!)
    tools = [{'name': n, 'description': t['desc'], 'input_schema': t['schema']}
             for n, t in system.tools.tools.items()]
    messages = [{'role': 'user', 'content': ctx}]
    for _ in range(max_steps):
        resp = client.messages.create(model=model, max_tokens=1024,
                                      tools=tools, messages=messages)
        if resp.stop_reason != 'tool_use':         # 模型完成
            return ''.join(b.text for b in resp.content if b.type == 'text')
        messages.append({'role': 'assistant', 'content': resp.content})
        results = []
        for b in resp.content:
            if b.type == 'tool_use':
                out = system.tools.dispatch({'name': b.name, 'input': b.input})  # 我们的 dispatch 原样可用!
                results.append({'type': 'tool_result', 'tool_use_id': b.id, 'content': out})
        messages.append({'role': 'user', 'content': results})
```

对应关系：MockLLM ↔ `messages.create`、`resp['type']=='tool_use'` ↔ `stop_reason=='tool_use'`、我们的 `select`/`inject`/`dispatch` 与三个注册表**原样适用**, `tool_registry` 的 schema 直接进 `tools=`。用 `make_llm` 的话, 无 key 时它自动回退 MockLLM——你的整套系统**永远跑得通**。这就是「亲手造一遍、结构对了、迁移只是换一行」的含义。

### 小结 —— 全课收口
- **完整 skills 系统 = 三注册表(skill/command/tool) 统一在『发现→选择→注入/调用』流水线 + 接到 agent 循环**。
- **选择**把潜在能力收敛成本轮实际能力；**组合**让多类能力协同完成单一能力做不到的事——这是完整系统相对单个零件的价值。
- **渐进披露贯穿到底**：三个注册表常驻轻量目录, 正文/schema 命中后才加载, 让系统可『拥有』大量能力却只为用到的付上下文成本。
- **agent 循环必须能终止**(max_steps 守卫), 端到端用世界状态判定任务真完成。
- **评测用数字说话**：召回/精度(选择质量)、完成率(最终判据)、token(渐进披露效果), 四维取平衡。
- **可迁移**：把 `MockLLM` 换成 `make_llm(model=...)`、内存件换真实管道, 你造的零件原样接到真实 Claude Code / MCP。

**全课五个模块到此拼成一台机器**：01 skill 加载 → 02 slash 命令 → 03 MCP server → 04 工具打包 → **05 完整系统**。你不依赖任何框架, 从零造出了一个可扩展 agent 的可扩展性子系统——既能用好 Claude Code, 也能在没有框架时自己搭一套。